# NYC Taxi Long-Trip Risk Model

This notebook runs the project pipeline from `src/nyc_taxi_long_trip_model.py`. It is intentionally thin so the source code remains version-controlled and reusable outside Colab.

## Setup

If this notebook is opened directly from GitHub, the first code cell clones the full repository into Colab so the `src/`, `reports/`, and `requirements.txt` files are available.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

repo_url = "https://github.com/soumithkumara/project_day_1.git"
repo_branch = "project_day_2"
repo_dir = Path("/content/project_day_1_project_day_2")

if repo_dir.exists() and (repo_dir / ".git").exists():
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", repo_branch], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "reset", "--hard", f"origin/{repo_branch}"], check=True)
else:
    if repo_dir.exists():
        shutil.rmtree(repo_dir)
    subprocess.run(
        ["git", "clone", "--branch", repo_branch, "--single-branch", repo_url, str(repo_dir)],
        check=True,
    )

os.chdir(repo_dir)
active_branch = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
active_commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"Working directory: {Path.cwd()}")
print(f"Active branch: {active_branch}")
print(f"Active commit: {active_commit}")
print("Repo files:", sorted(path.name for path in Path.cwd().iterdir()))

In [ ]:
!pip install -r requirements.txt

In [ ]:
from src.nyc_taxi_long_trip_model import (
    DEFAULT_DATA_URL,
    DEFAULT_MODEL_DIR,
    DEFAULT_RAW_PATH,
    DEFAULT_REPORT_DIR,
    PipelineConfig,
    run_pipeline,
)

config = PipelineConfig(
    data_url=DEFAULT_DATA_URL,
    raw_path=DEFAULT_RAW_PATH,
    report_dir=DEFAULT_REPORT_DIR,
    model_dir=DEFAULT_MODEL_DIR,
    sample_size=120_000,
    force_download=False,
)

result = run_pipeline(config)
result

## Preprocessing Audit

The pipeline records every major cleaning step. The table below shows how many rows existed before each step, how many remained after the step, how many were removed, and what percent of the original raw data remained.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

audit = pd.read_csv("reports/preprocessing_audit.csv")
display(audit)

raw_rows = int(audit.loc[audit["step"] == "01_raw_load", "rows_after"].iloc[0])
clean_rows = int(audit.loc[audit["step"] == "12_feature_engineering", "rows_after"].iloc[0])
sample_rows = int(audit.loc[audit["step"] == "13_model_sample", "rows_after"].iloc[0])
display(Markdown(
    f"**Summary:** The raw file started with **{raw_rows:,} rows**. "
    f"After cleaning, **{clean_rows:,} realistic trips** remained. "
    f"The model used a deterministic **{sample_rows:,}-row sample** so the notebook runs quickly and reproducibly."
))

## Data Changes and Feature Engineering

This table explains exactly what new fields were created or converted, where each came from, and why it matters for the model.

In [ ]:
feature_changes = pd.read_csv("reports/feature_engineering_summary.csv")
display(feature_changes)

target_distribution = pd.read_csv("reports/target_distribution.csv")
display(Markdown("### Target Distribution After Preprocessing"))
display(target_distribution)

In [ ]:
import json
from pathlib import Path

metrics_path = Path("reports/metrics.json")
metrics = json.loads(metrics_path.read_text())
print(json.dumps(metrics, indent=2))

display(Markdown("### Classification Report"))
print(metrics["classification_report"])

## Model Metrics and Outputs

The model is evaluated against a most-frequent baseline. Because long trips are the minority class, recall and F1 are more informative than accuracy alone.

In [ ]:
import pandas as pd
from IPython.display import display

display(pd.read_csv("reports/model_metrics.csv"))
display(pd.read_csv("reports/tuning_results.csv"))
display(pd.read_csv("reports/model_feature_importance.csv"))

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

figure_dir = Path("reports/figures")
for figure_path in sorted(figure_dir.glob("*.png")):
    title = figure_path.stem.replace("_", " ").title()
    display(Markdown(f"### {title}"))
    display(Image(filename=str(figure_path)))